In [1]:
"""
Unit tests for NumPy Stock Analyzer concepts.
Run with: python -m pytest tests/ -v
"""

import numpy as np
import pytest


# ── Helpers (mirrors logic in analyzer.py) ────────────────────────────────────

def make_prices(seed=42, days=252, n_stocks=5):
    np.random.seed(seed)
    base = np.array([1500, 1700, 1100, 380, 1800], dtype=np.float32)
    returns = np.random.normal(0.0005, 0.015, (days, n_stocks)).astype(np.float32)
    prices = np.zeros((days, n_stocks), dtype=np.float32)
    prices[0] = base
    for d in range(1, days):
        prices[d] = prices[d - 1] * (1 + returns[d])
    return prices, returns


# ── Tests ─────────────────────────────────────────────────────────────────────

class TestArrayProperties:
    def test_shape(self):
        prices, _ = make_prices()
        assert prices.shape == (252, 5)

    def test_dtype_is_float32(self):
        prices, _ = make_prices()
        assert prices.dtype == np.float32

    def test_memory_savings(self):
        prices, _ = make_prices()
        assert prices.nbytes < prices.astype(np.float64).nbytes

    def test_prices_positive(self):
        prices, _ = make_prices()
        assert np.all(prices > 0)


class TestSlicingAndIndexing:
    def test_first_row_equals_base(self):
        prices, _ = make_prices()
        base = np.array([1500, 1700, 1100, 380, 1800], dtype=np.float32)
        np.testing.assert_array_equal(prices[0], base)

    def test_column_slice_shape(self):
        prices, _ = make_prices()
        col = prices[:, 0]        # first stock, all days
        assert col.shape == (252,)

    def test_subarray_slice(self):
        prices, _ = make_prices()
        sub = prices[0:10, 0:3]  # first 10 days, first 3 stocks
        assert sub.shape == (10, 3)

    def test_fancy_indexing(self):
        arr = np.array([10, 20, 30, 40, 50])
        selected = arr[[0, 2, 4]]
        np.testing.assert_array_equal(selected, [10, 30, 50])


class TestBooleanMasking:
    def test_crash_mask_count(self):
        _, returns = make_prices()
        crash_mask = returns < -0.03
        assert crash_mask.sum() > 0      # there should be some crashes
        assert crash_mask.dtype == bool

    def test_mask_filters_correctly(self):
        arr = np.array([10, 25, 5, 40, 15])
        result = arr[arr > 20]
        np.testing.assert_array_equal(result, [25, 40])


class TestVectorizationAndBroadcasting:
    def test_cumulative_return_shape(self):
        prices, _ = make_prices()
        cum = (prices[-1] - prices[0]) / prices[0] * 100
        assert cum.shape == (5,)

    def test_normalisation_mean_near_zero(self):
        prices, _ = make_prices()
        mean_p = np.mean(prices, axis=0)
        std_p = np.std(prices, axis=0)
        normalised = (prices - mean_p) / std_p
        np.testing.assert_allclose(np.mean(normalised, axis=0),
                                   np.zeros(5), atol=1e-5)

    def test_normalisation_std_near_one(self):
        prices, _ = make_prices()
        mean_p = np.mean(prices, axis=0)
        std_p = np.std(prices, axis=0)
        normalised = (prices - mean_p) / std_p
        np.testing.assert_allclose(np.std(normalised, axis=0),
                                   np.ones(5), atol=1e-5)

    def test_broadcast_scalar_add(self):
        arr = np.array([1, 2, 3])
        result = arr + 10
        np.testing.assert_array_equal(result, [11, 12, 13])


class TestMathFunctions:
    def test_correlation_matrix_shape(self):
        _, returns = make_prices()
        corr = np.corrcoef(returns.T)
        assert corr.shape == (5, 5)

    def test_correlation_diagonal_is_one(self):
        _, returns = make_prices()
        corr = np.corrcoef(returns.T)
        np.testing.assert_allclose(np.diag(corr), np.ones(5), atol=1e-5)

    def test_argmax_argmin(self):
        arr = np.array([3, 1, 4, 1, 5, 9, 2, 6])
        assert np.argmax(arr) == 5
        assert np.argmin(arr) == 1

    def test_reshape_flatten(self):
        arr = np.arange(6)
        mat = arr.reshape(2, 3)
        assert mat.shape == (2, 3)
        assert mat.flatten().shape == (6,)

    def test_cumsum(self):
        arr = np.array([1, 2, 3, 4])
        np.testing.assert_array_equal(np.cumsum(arr), [1, 3, 6, 10])

ModuleNotFoundError: No module named 'pytest'